In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Question 1
Find the effect in dataset 12.1, given that Group = 1 is the treatment group and Time > 0 is the treatment time. 

- A: 0.5

- B: 1

- C: 1.5

- D: 2

In [2]:
df = pd.read_csv("../data/homework_12.1.csv")
df.head()

,Y,Time,Group
0,-0.232900,-1.193204,0
1,2.848846,-1.607748,1
2,0.550209,-0.269793,0
3,2.198280,7.743730,0
4,4.111044,-4.244359,1


**Way 1**

In [3]:
df1 = df.copy()
df1["Post"] = (df1["Time"] > 0).astype(int)
df1["Treated"] = df1["Group"] * df1["Post"]

model1 = smf.ols("Y ~ Group + Post + Treated",data=df1).fit()

print(model1.params)

Intercept    0.980217
Group        2.042010
Post         1.512989
Treated      0.944010
dtype: float64


**Way 2**

In [4]:
df2 = df1.copy()
df2["Post"] = (df2["Time"] > 0).astype(int)

model2 = smf.ols("Y ~ Group * Post",data=df2).fit()

print(model2.params)

Intercept     0.980217
Group         2.042010
Post          1.512989
Group:Post    0.944010
dtype: float64


**Way 3**

In [5]:
df3 = df1.copy()
df3["Post"] = (df3["Time"] > 0).astype(int)

model3 = smf.ols("Y ~ C(Group) + Post + C(Group):Post",data=df3).fit()

print(model3.params)

Intercept             0.980217
C(Group)[T.1]         2.042010
Post                  1.512989
C(Group)[T.1]:Post    0.944010
dtype: float64


Answer:  ‌‌B 
______

## Question 2
With dataset 12.2, run a test to check prior trends. To do this, run a linear regression on the data before Time = 0, with an interaction term equal to Group x Time.   
Which is the closest to the t-value of the interaction term? 

- A: 0.02

- B: 1

- C: 5

- D: 25

In [6]:
df_2 = pd.read_csv("../data/homework_12.2.csv")
df_2.head()

,Y,Time,Group
0,0.283614,-3.614416,0
1,3.181495,-3.064196,0
2,1.401619,-4.109261,0
3,2.019632,-0.706696,1
4,1.032628,-1.806227,0


In [8]:
df_selected = df_2.loc[df_2["Time"] < 0, :]
df_selected.head()

,Y,Time,Group
0,0.283614,-3.614416,0
1,3.181495,-3.064196,0
2,1.401619,-4.109261,0
3,2.019632,-0.706696,1
4,1.032628,-1.806227,0


In [24]:
# Fit the model with interaction term
model4 = smf.ols("Y ~ Group*Time",data=df_selected).fit()

print(f" Interaction term coefficient: {model4.params['Group:Time']:.5f}")
print(f" Interaction term p-value: {model4.pvalues['Group:Time']:.7f}")
print(f" Interaction term t-value: {model4.tvalues['Group:Time']:.5f}")
print(f" Interaction term standard error: {model4.bse['Group:Time']:.5f}")
print(f" Time range: {df_selected['Time'].min()} to {df_selected['Time'].max()}")

 Interaction term coefficient: 0.04640
 Interaction term p-value: 0.0000001
 Interaction term t-value: 5.32938
 Interaction term standard error: 0.00871
 Time range: -9.999416398 to -0.001149172


Answer:   C
______

## Question 3
With the following, make an error such that the correlation (np.corrcoef) between the error at time t and time t + 1 is 0.80 on average, the mean error is 0, and the standard deviation of the error is 1. (These may be approximate.) There are 10,000 items. Given: 

X = this error    
    
Y = 2 * X + this error (recalculated with new randomness relative to the previous line)     

The standard error of the estimated X coefficient computed via Ordinary Least Squares (OLS) basic standard error should be about 0.01. Then the standard error computed via simulation (the standard deviation of the estimated coefficient over many trials) is closest to: 

Note: you might consider Googling / asking ChatGPT how to generate an autocorrelated time series with a specific correlation and standard deviation. 

- A: 0.0025

- B: 0.005

- C: 0.01

- D: 0.02

In [28]:
n = 10000
noise = np.random.normal(0, 1, n)

error = np.zeros(n)
error[0] = noise[0]

for t in range(1, n):
    error[t] = 0.8 * error[t-1] + 0.6 * noise[t]

print("Mean:", error.mean())
print("SD:", error.std())
print("Correlation:", np.corrcoef(error[:-1], error[1:])[0, 1])

Mean: -0.040078171535737894
SD: 0.9860863047166816
Correlation: 0.7938939303069256


In [29]:
X = error
noise2 = np.random.normal(0, 1, n)

error2 = np.zeros(n)
error2[0] = noise2[0]

for t in range(1, n):
    error2[t] = 0.8 * error2[t-1] + 0.6 * noise2[t]

Y = 2 * X + error2

In [31]:
df = pd.DataFrame({"X": X, "Y": Y})

model = smf.ols("Y ~ X", data=df).fit()

print(model.params)
print("******")
print(model.bse)

Intercept    0.055336
X            1.957507
dtype: float64
******
Intercept    0.009925
X            0.010057
dtype: float64


In [32]:
coefficients = []
ols_se = []

for i in range(1000):

    # 1. Generate X
    noise1 = np.random.normal(0, 1, n)
    X = np.zeros(n)
    X[0] = noise1[0]

    for t in range(1, n):
        X[t] = 0.8 * X[t-1] + 0.6 * noise1[t]

    # 2. Generate a new independent error
    noise2 = np.random.normal(0, 1, n)
    error2 = np.zeros(n)
    error2[0] = noise2[0]

    for t in range(1, n):
        error2[t] = 0.8 * error2[t-1] + 0.6 * noise2[t]

    # 3. Generate Y
    Y = 2 * X + error2

    # 4. Run OLS
    df_sim = pd.DataFrame({"X": X, "Y": Y})
    model_sim = smf.ols("Y ~ X", data=df_sim).fit()

    # 5. Save coefficient and OLS SE
    coefficients.append(model_sim.params["X"])
    ols_se.append(model_sim.bse["X"])

In [33]:
print("Mean coefficient:", np.mean(coefficients))

print("Average OLS SE:", np.mean(ols_se))

print("Simulation SE:", np.std(coefficients, ddof=1))

Mean coefficient: 1.999113925432784
Average OLS SE: 0.009997896024026483
Simulation SE: 0.022030490087728147


Answer:    D
______

## Question 4

Find the variance inflation factor of X1:   

- A: 2

- B: 4

- C: 8

- D: 16

In [25]:
np.random.seed(0) 

X1 = np.random.normal(0, 1, 1000) 

X2 = np.random.normal(0, 1, 1000) + X1 

X3 = np.random.normal(0, 1, 1000) + 2 * X2 

In [27]:
df5 = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})
model5 = smf.ols("X1 ~ X2 + X3", data=df5).fit()
model5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     X1   R-squared:                       0.495
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     487.7
Date:                Sun, 22 Nov 2026   Prob (F-statistic):          1.95e-148
Time:                        21:25:47   Log-Likelihood:                -1064.7
No. Observations:                1000   AIC:                             2135.
Df Residuals:                     997   BIC:                             2150.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0301      0.022     -1.352      0.177      -0.074       0.014
X2             0.5472      0.049     11.177      0.000       0.451       0.643
X3            -0.0188      0.023     -0.806      0.420      -0.065       0.027
==============================================================================
Omnibus:                        0.162   Durbin-Watson:                   2.046
Prob(Omnibus):                  0.922   Jarque-Bera (JB):                0.088
Skew:                           0.008   Prob(JB):                        0.957
Kurtosis:                       3.043   Cond. No.                         7.60
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [35]:
r2 = model5.rsquared
vif = 1 / (1 - r2)

print("R-squared:", r2)
print("VIF of X1:", vif)

R-squared: 0.4945344638582029
VIF of X1: 1.9783742480901254


Answer: A  
___________